# 04 · Ray on Vertex AI

Run the **Python-runtime models on a Ray-on-Vertex cluster** in parallel with the in-BigQuery native track — the same `main.run(cfg)` entrypoint as every other demo, dispatched to Ray by `cfg.python_runtime="ray"`. A fixed-size cluster is sized to the run's fan-out (no autoscaling — D17), the job runs on it, and the cluster is torn down in a `finally` so nothing bills after the run.

> **Run this notebook *inside* GCP** — a **Vertex AI Workbench** or **Colab Enterprise** notebook. Submitting a Ray Job goes through the cluster's dashboard proxy host (`*.aiplatform-training.googleusercontent.com`); a workstation outside the project's network gets a `524` upstream-timeout on the `JobSubmissionClient` handshake even though the cluster itself comes up fine. From an in-GCP kernel that host is reachable. (Cluster *create*, the BQ natives, and *teardown* all work from anywhere — only the **job-submission** hop needs in-network reachability.)

## Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this clones or updates the repo so you're on the latest `src/`. **Skip it in a local clone** — it's a no-op guarded on the package already being importable. The `[ray]` extra must be installed in the kernel (`pip install -e 'scale-forecasting[ray]'`).

In [ ]:
# Cloud bootstrap: clone + editable-install so `import scale_forecasting` resolves.
# Harmless locally — if the package already imports, we do nothing.
import importlib.util
import os
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[ray]"], check=True)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Resolve the deployment + Ray infra (live GCP)

`Settings.resolve()` reads the `SF_*` environment — the *same* identity every writer uses (G1). Required: `SF_PROJECT_ID`, `SF_CONNECTION`, `SF_WAREHOUSE_URI`; `SF_DATASET_ID`/`SF_REGION` default.

The Ray submitter needs a little more (beyond that identity), resolved by `RayInfra`: **`SF_COMPUTE_SA`** and **`SF_CODE_BUCKET`** are required; `SF_RAY_NETWORK` is optional (unset → the public endpoint). Every value comes straight from `terraform output` in `terraform/main`.

In [ ]:
from google.cloud import bigquery

from scale_forecasting.ray_submit import RayInfra
from scale_forecasting.settings import Settings

settings = Settings.resolve()
infra = RayInfra.resolve()  # raises naming the first missing SF_COMPUTE_SA / SF_CODE_BUCKET
client = bigquery.Client(project=settings.project_id)
DATASET = settings.dataset_ref
print("deployment:", DATASET, "region:", settings.region)
print("ray infra: compute_sa set:", bool(infra.compute_sa), "| code_bucket:", infra.code_bucket)

## Review helpers

Registry rows written through the Storage Write API are *async-visible*, so we poll the leaderboard briefly until a run's models show up. `leaderboard(run_id)` returns one row per model — `compute_engine` splits Ray / BigQuery / ensemble — and `run_summary(run_id)` is the header roll-up.

In [ ]:
import time

import pandas as pd


def _query(sql, run_id):
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", run_id)]
        ),
    )
    return job.result().to_dataframe()


def leaderboard(run_id, expect_models=None, tries=12, pause=3.0):
    """Poll v_model_leaderboard until `expect_models` all appear (or give up), newest metrics."""
    sql = (
        f"SELECT model_type, compute_engine, n_cells, mean_wape, mean_mae "
        f"FROM `{DATASET}.v_model_leaderboard` WHERE run_id=@run_id "
        f"ORDER BY mean_wape"
    )
    df = pd.DataFrame()
    for _ in range(tries):
        df = _query(sql, run_id)
        if expect_models is None or set(df["model_type"]) >= set(expect_models):
            break
        time.sleep(pause)
    return df


def run_summary(run_id):
    sql = f"SELECT * FROM `{DATASET}.v_run_summary` WHERE run_id=@run_id"
    return _query(sql, run_id)

## Run on Ray (CPU-only)

`configs/ray_cpu_demo.json` is `python_runtime="ray"` with `compute.use_gpu=false` and no `neuralprophet`, so `plan_cluster` sizes the GPU pool to **zero** and provisions only the CPU worker pool — no T4 quota needed. `main.run(cfg)` sizes + creates the cluster, runs the stats models (`theta`, `holtwinters`) on Ray in parallel with the natives (`arima_plus`, `arima_plus_xreg`, `timesfm`) in BigQuery under one shared `run_id`, then tears the cluster down.

> Expect **~15–25 min**: fixed-cluster stand-up dominates. `ray_regions` in the config lets the launcher hop US regions if one transiently stocks out on capacity.
>
> **GPU showpiece.** To add the fractional-T4 NeuralProphet cell, use `configs/ray_gpu_demo.json` instead (`use_gpu:true`, `neuralprophet` in `models`) — identical call, one config apart (G2). That path needs `NVIDIA_T4_GPUS` quota in one of the `ray_regions`.

In [ ]:
from scale_forecasting import main
from scale_forecasting.config import load_config
from scale_forecasting.registry.ids import make_run_id

cfg = load_config("configs/ray_cpu_demo.json")
run_id = make_run_id(cfg)
print("run_id:", run_id, "| models:", cfg.models, "| runtime:", cfg.python_runtime)

returned = main.run(cfg)
assert returned == run_id
print("ray run complete:", run_id)

## Review — both engines on one leaderboard

The stats models ran on Ray (`compute_engine='ray'`); the natives in BigQuery (`compute_engine='bigquery'`) — all under one `run_id`, ranked by `mean_wape`.

In [ ]:
leaderboard(run_id, expect_models=cfg.models)

In [ ]:
run_summary(run_id)

## What the run recorded — the fixed cluster sizing

The header's `job_telemetry` audits the sizing decision that actually ran: `runtime='ray'`, the fixed per-pool node counts, and (on the GPU path) the calibrated `sizing_gpu_fraction` + `accelerator_type`. On this CPU-only run `gpu_node_count` is `0` — the whole cluster is the CPU worker pool.

In [ ]:
import json

hdr = _query(
    f"SELECT job_telemetry FROM `{DATASET}.run_registry` WHERE run_id=@run_id", run_id
)
json.loads(hdr["job_telemetry"].iloc[0]) if not hdr.empty else "(header not visible yet)"